# Day.12 LAG
 - RAG chain 구성

## 준비
- 환경 변수 준비하기
- llm-> 폴백구조 (이가 없으면 잇몸으로..... Openai -> Gemini)

## Indexing
- 데이터를 가져오기 -> Dataloader를 통해서 Documnets
- chunking -> TextSplitter 작업 (chunk size, overlab -->)
- embedding -> 문자 -> 숫자( 의미가 부여된 숫자 ---> embedding모델을 통해서)
- vector store -> 저장(embedding된 데이터, 의미가 유사한 데이터를 쉽고, 빠르게 찾아오기)
    - chroma, FAISS, pinecone, firebase, supabase(PGvector)

In [1]:
from dotenv import load_dotenv
import os, json


load_dotenv(override=True)

openai_key = os.getenv("OPENAI_API_KEY")
openai_default_model = os.getenv("OPENAI_DEFAULT_MODEL", "gpt-6-luna")
gemini_key = os.getenv("GOOGLE_API_KEY")
gemini_default_model = os.getenv("GEMINI_DEFAULT_MODEL", "gemini-4-hydrogen")
openai_embedding_model = os.getenv("EMBEDDING_MODEL")

In [2]:
##############################################
################ fallback 구조 ################
##############################################
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser


g_llm = init_chat_model(f'google_genai:{gemini_default_model}')
o_llm = init_chat_model(f'openai:{openai_default_model}')

with_fallbacks_llm = o_llm.with_fallbacks([g_llm])

In [3]:
#1. 데이터 가져오기
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

loader = TextLoader('data/data.txt', encoding='utf-8')
documents = loader.load()
# print(documents[0].page_content)
documents[0].page_content = documents[0].page_content.replace("\n\n", "\n")
text_spliiter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
docs = text_spliiter.split_documents(documents)
for i, doc in enumerate(docs):
    # print(doc)
    print(f"Chunk {i+1}: {doc.page_content}")

embeddings = OpenAIEmbeddings(model=openai_embedding_model)

vector_store = Chroma.from_documents(
    documents = docs,
    embedding= embeddings,
    ids = [f'Capital-{i}' for i in range(len(docs))],
    collection_name="my_Captial",
    persist_directory="chroma_db"
)

C:\Users\Admin\AppData\Local\Temp\ipykernel_19788\2380912143.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Chunk 1: 프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.
Chunk 2: 독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.
Chunk 3: 일본의 수도는 도쿄이다. 도쿄는 기술과 문화의 중심지로, 애니메이션과 음식 문화가 발달해 있다.
Chunk 4: 영국의 수도는 런던이다. 런던은 전통과 현대가 공존하는 도시로, 빅벤과 타워 브리지가 유명하다.
Chunk 5: 이탈리아의 수도는 로마이다. 로마는 고대 역사의 숨결이 살아 숨 쉬는 도시로, 콜로세움과 트레비 분수가 유명하다.
Chunk 6: 미국의 수도는 워싱턴 D.C.이다. 워싱턴 D.C.는 세계 정치와 외교의 중심지로, 백악관과 링컨 기념관이 유명하다.
Chunk 7: 호주의 수도는 캔버라이다. 캔버라는 계획적으로 조성된 행정 도시이며, 국회의사당과 아름다운 인공 호수인 벌리 그리핀 호수가 유명하다.
Chunk 8: 스페인의 수도는 마드리드이다. 마드리드는 정열과 예술의 도시로, 프라도 미술관과 화려한 왕궁이 유명하다.


In [4]:
retriever = vector_store.as_retriever(search_type="similarity", 
                          search_kwargs={"k":2})

for answer in retriever.invoke("프랑스의 수도는?"):
    print(answer)

page_content='프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.' metadata={'source': 'data/data.txt'}
page_content='독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.' metadata={'source': 'data/data.txt'}


In [5]:
for doc, score in vector_store.similarity_search_with_relevance_scores('프랑스의 수도는?', k=2):
    print(score, doc.page_content)

0.48673740685512146 프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.
0.2394601186209172 독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.


In [6]:
# Similarity_score_threshold

t_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k":2,
                   "score_threshold": 0.4,
                  },
)

for answer in t_retriever.invoke('프랑스의 수도는?'):
    print(answer)

page_content='프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.' metadata={'source': 'data/data.txt'}


In [8]:
# mmr 

m_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k":2,
                   "score_threshold": 0.4,
                   "fetch_k":5,
                   },
)

for answer in t_retriever.invoke('프랑스의 수도는?'):
    print(answer)

page_content='프랑스의 수도는 파리이다. 파리는 유럽에서 가장 인기 있는 관광 도시 중 하나로, 에펠탑과 루브르 박물관이 유명하다.' metadata={'source': 'data/data.txt'}
page_content='독일의 수도는 베를린이다. 베를린은 역사적으로 중요한 도시이며, 베를린 장벽으로 유명하다.' metadata={'source': 'data/data.txt'}


# LAG Chain

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough



prompt = PromptTemplate.from_template(
    '''
다음 문서를 참고해서 질문에 답하세요.
문서: {context}
질문: {question}
'''
)

def format_docs(docs):
    return '\n'.join(doc.page_content for doc in docs)

rag_chain = (
    {'context':m_retriever| format_docs, 'question':RunnablePassthrough()}
    |prompt
    |o_llm
    |StrOutputParser())

ret = rag_chain.invoke('프랑스의 수도는?')
ret

'파리입니다.'

In [ ]:
## RAG Chain을 만들기 위해 공통 

In [46]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

rag_prompt = ChatPromptTemplate.from_template(
'''
다음 문서를 참고해서 질문에 답해주세요. 문서에 없는 내용은 지어내지 말고 "그런 내용 없습니다"라고 답해줘
문서: {context}
질문: {question}
'''
)

def format_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

new_retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={"k":2,
                   "score_threshold":0.4,
                   "fetch_k":5,
                   },
)

def rag_chain(retriever, llm=o_llm):
    return (
        {"context": retriever | format_docs, "question":RunnablePassthrough()}
        | rag_prompt
        | llm
        | StrOutputParser()
    )



In [16]:
new_rag_chain = rag_chain(new_retriever, o_llm)
new_rag_chain.invoke('프랑스의 수도는 어디?')

'파리입니다.'

In [17]:
from langchain_core.documents import Document

lecture_docs = [
    Document(page_content="RAG는 검색 증강 생성 기법이다.", metadata={"source": "lecture"}),
    Document(page_content="Chroma는 RAG에 특화된 오픈소스 벡터DB다.", metadata={"source": "lecture"}),
    Document(page_content="임베딩은 텍스트를 벡터로 변환하는 과정이다.", metadata={"source": "lecture"}),
    Document(page_content="챗봇은 사용자 질문에 답변하는 프로그램이다.", metadata={"source": "lecture"}),
]
print(len(lecture_docs), "개 문서 준비")

4 개 문서 준비


In [22]:
chroma_store = Chroma.from_documents(
    documents=lecture_docs,
    embedding=embeddings,
    ids= [f'lecture-{i+1}' for i in range(len(lecture_docs))],
    collection_name = "lecture_collection",
    persist_directory= "lecture_chroma"
)

In [23]:
lecture_retriever = chroma_store.as_retriever(search_kwars={"k":2})

rag_chain = rag_chain(lecture_retriever, o_llm)
rag_chain.invoke('RAG가 뭐니?')

'RAG는 검색 증강 생성 기법입니다.'

In [48]:
import chromadb
from langchain_chroma import Chroma




PERSIST_DIR = "./chroma_capital_db"
COLLECTION_NAME = "capital_collection"




client = chromadb.PersistentClient(path=PERSIST_DIR)
existing = [c.name for c in client.list_collections()]
vectorstore = Chroma(client=client, collection_name=COLLECTION_NAME, embedding_function=embeddings)


# print(vectorstore)
# if COLLECTION_NAME in existing and vectorstore.get()["ids"]:
#     print(f"[로드] '{COLLECTION_NAME}' 컬렉션 사용 ({len(vectorstore.get()['ids'])}개)")
# else:
#     vectorstore.add_documents(lecture_docs, ids=[f"lecture-{i}" for i in range(len(lecture_docs))])
#     print(f"[생성] '{COLLECTION_NAME}' 컬렉션에 {len(lecture_docs)}개 저장")

# for doc in vectorstore.similarity_search("Chroma가 뭐야?", k=2):
#     print(doc.page_content)

load_retriever = vectorstore.as_retriever(search_kwargs={'k':2})
# print(load_retriever)
invoke_rag_chain = rag_chain(load_retriever)
invoke_rag_chain.invoke('프랑스의 수도는??')

'파리입니다.'

## webbaseloader

- 위키피디아 뉴턴

In [ ]:
from langchain_community.document_loaders import WebBaseLoader

url = 'https://en.wikipedia.org/wiki/Isaac_Newton'
loader = WebBaseLoader(url)
wiki_documents = loader.load()
print(wiki_documents[0].page_content)





Isaac Newton - Wikipedia






























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools






Donate


Create account


Log in





























Contents
move to sidebar
hide




(Top)





1
Early life




Toggle Early life subsection





1.1
The King's School








1.2
University of Cambridge








1.3
Revision of Geographia Generalis










2
Scientific studies




Toggle Scientific studies subsection





2.1
Mathematics








2.2
Optics








2.3
Philosophiæ Naturalis Principia Mathematica








2.4
Other significant work








2.5
Philosophy of science










3
Later life




Toggle La

In [53]:
#불필요한 공백 제거
import re
def clean_text(text: str) -> str:
    text = re.sub(r"[ \t\xa0]+", " ", text)     # 연속 공백·탭·특수공백(&nbsp;) → 공백 1개
    text = re.sub(r"\n\s*\n+", "\n", text)       # 빈 줄·공백만 있는 줄 → 줄바꿈 1개
    return text.strip()                          # 앞뒤 공백 제거
# WebBaseLoader로 불러온 documents에 적용
def clean_documents(documents):    
    for doc in documents:
        doc.page_content = clean_text(doc.page_content)
    return documents    
wiki_documents = clean_documents(wiki_documents)
print("제목:", wiki_documents[0].metadata.get("title"))
print("글자 수:", len(wiki_documents[0].page_content))
print(wiki_documents[0].page_content[:500])

제목: Isaac Newton - Wikipedia
글자 수: 185894
Isaac Newton - Wikipedia
Jump to content
Main menu
Main menu
move to sidebar
hide
 Navigation
Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us
 Contribute
HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages
Search
Search
Appearance
Donate
Create account
Log in
Personal tools
Donate
Create account
Log in
Contents
move to sidebar
hide
(Top)
1
Early life
Toggle Early life subsection
1.1
The King's School
1.2
University of Cambridge
1.3
Revision of Geographi


In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import re



url = 'https://en.wikipedia.org/wiki/Isaac_Newton'
loader = WebBaseLoader(url)
wiki_documents = loader.load()
# print(documents[0].page_content)
def clean_text(text: str) -> str:
    text = re.sub(r"[ \t\xa0]+", " ", text)    
    text = re.sub(r"\n\s*\n+", "\n", text)      
    return text.strip()                         
def clean_documents(documents):    
    for doc in documents:
        doc.page_content = clean_text(doc.page_content)
    return documents    
wiki_documents = clean_documents(wiki_documents)

text_spliiter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
wiki_docs = text_spliiter.split_documents(wiki_documents)
for i, doc in enumerate(wiki_docs):
    # print(doc)
    print(f"Chunk {i+1}: {doc.page_content}")

embeddings = OpenAIEmbeddings(model=openai_embedding_model)

wiki_store = Chroma.from_documents(
    documents = wiki_docs,
    embedding= embeddings,
    ids = [f'wiki-{i}' for i in range(len(wiki_docs))],
    collection_name="newton_collection",
    persist_directory="wiki_chroma_1"
)

Chunk 1: Isaac Newton - Wikipedia
Jump to content
Main menu
Main menu
move to sidebar
hide
 Navigation
Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us
 Contribute
Chunk 2: Contribute
HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages
Search
Search
Appearance
Donate
Create account
Log in
Personal tools
Donate
Create account
Log in
Contents
Chunk 3: Log in
Contents
move to sidebar
hide
(Top)
1
Early life
Toggle Early life subsection
1.1
The King's School
1.2
University of Cambridge
1.3
Revision of Geographia Generalis
2
Scientific studies
Chunk 4: Scientific studies
Toggle Scientific studies subsection
2.1
Mathematics
2.2
Optics
2.3
Philosophiæ Naturalis Principia Mathematica
2.4
Other significant work
2.5
Philosophy of science
3
Later life
Chunk 5: 3
Later life
Toggle Later life subsection
3.1
Royal Mint
3.2
Knighthood
3.3
Wealth
3.4
Death
4
Personality
5
Theology and chronology
Toggle Theology and chronology subsection
5.1
Religious views
5.2

In [65]:
def wiki_rag_chain(retriever,prompt, llm=o_llm):
    return (
        {"context": retriever | format_docs, "question":RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

In [66]:
### Step 3. RAG 체인 만들고 질문하기

wiki_retriever = wiki_store.as_retriever(search_kwargs={"k": 2})

basic_prompt = ChatPromptTemplate.from_template(
    "다음 컨텍스트를 참고해서 질문에 답해줘.\n\n컨텍스트: {context}\n\n질문: {question}"
)
wiki_chain = wiki_rag_chain(wiki_retriever, basic_prompt)

questions = [
    "뉴턴의 본명은?",
    "뉴턴는 지금 몇 살?",
    "뉴턴의 데뷔곡은? 몇 년도에 데뷔했니?",
    "뉴턴의 첫 번째 드라마는?",
    "뉴턴의 가장 최근 드라마는?",
]

for q in questions:
    print("Q:", q)
    print("A:", wiki_chain.invoke(q), "\n")

Q: 뉴턴의 본명은?
A: 뉴턴의 본명은 아이작 뉴턴(Isaac Newton)입니다. 

Q: 뉴턴는 지금 몇 살?
A: 뉴턴은 1727년에 **84세로 사망했습니다**. 지금까지 살아 있었다면 2026년 11월 기준 **383세**입니다. 

Q: 뉴턴의 데뷔곡은? 몇 년도에 데뷔했니?
A: 뉴턴 파크너(Newton Faulkner)를 말하는 거라면, 데뷔곡은 **「Dream Catch Me」**이고 **2007년**에 데뷔했습니다. 

Q: 뉴턴의 첫 번째 드라마는?
A: 제시된 문맥에는 뉴턴의 첫 번째 드라마에 관한 정보가 없습니다. 뉴턴은 드라마 작품을 쓴 것으로 알려져 있지 않습니다. 

Q: 뉴턴의 가장 최근 드라마는?
A: 제공된 컨텍스트에는 뉴턴이 출연하거나 다룬 드라마에 관한 정보가 없습니다. 뉴턴을 “마지막 마술사”라고 묘사한 인용문만 있습니다. 



In [ ]:
# pdf lodaer
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import re

def clean_tokenized_text(text: str) -> str:
    # 1. 모든 줄바꿈을 일단 공백 하나로 합침
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. 문단/항목 기호(◇, △) 앞에는 줄바꿈(\n\n)을 넣어 청킹 기준점 확보
    text = re.sub(r'\s*(◇|△)\s*', r'\n\n\1 ', text)
    
    # 3. 괄호 및 따옴표 안쪽의 불필요한 공백 제거: ( 서울 = 뉴스 1 ) -> (서울=뉴스1)
    text = re.sub(r'\(\s+', '(', text)
    text = re.sub(r'\s+\)', ')', text)
    text = re.sub(r'\s*=\s*', '=', text)
    text = re.sub(r"'\s+([^']+?)\s+'", r"'\1'", text)
    text = re.sub(r'"\s+([^"]+?)\s+"', r'"\1"', text)
    
    # 4. 문장 부호(., ,) 앞의 불필요한 공백 제거
    text = re.sub(r'\s+([.,])', r'\1', text)
    
    # 5. 숫자와 단위/조사 사이 공백 제거 (예: 6 월 -> 6월, 23 명 -> 23명, 1 년 -> 1년)
    text = re.sub(r'(\d)\s+([가-힣])', r'\1\2', text)
    
    # 6. 한글 어간과 분리된 조사/어미 결합 (예: 한국 축구대표팀 이 -> 한국 축구대표팀이)
    # 자주 분리되는 한국어 조사/어미 패턴
    josa_pattern = r'(?<=[가-힣\)])\s+(은|는|이|가|을|를|의|에|에서|에게|로|으로|와|과|도|만|까지|부터|마저|조차|이나|나|이며|며|고|다고|라고|이라고|했다|됐다|밝혔다|말했다|설명했다|있다|없다|중|후|전|등|안|밖)(?=[\s.,\)\"\']|$)'
    # 여러 번 연달아 붙을 수 있으므로 2회 반복 적용 (예: 선수 측 의 -> 선수측의)
    for _ in range(2):
        text = re.sub(josa_pattern, r'\1', text)
        
    # 7. 문장 끝(. 또는 ." 또는 .') 뒤에 줄바꿈(\n\n)을 추가하여 RecursiveCharacterTextSplitter가 문장 단위로 자르기 좋게 만듦
    text = re.sub(r'(\.(?:["\'])?)\s+', r'\1\n\n', text)
    
    return text.strip()

manual_path = "data/2026 FIFA북중미 월드컵 아시아지역 2차 예선 6월 A대표팀 선수 소집 명단 발표.pdf"

pages = PyPDFLoader(manual_path).load()
# print(pages[0].page_content)
pages[0].page_content = clean_tokenized_text(pages[0].page_content)
# print(pages[0].page_content)


manual_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""] # \n\n(문단/문장) 기준으로 우선 분할
)

manual_chunks = manual_splitter.split_documents(pages)
print("청크 수:", len(manual_chunks))



청크 수: 7


In [101]:
print(pages[0].page_content)

(서울=뉴스 1) 안영준 기자='임시 사령탑' 김도훈 감독이 이끄는 한국 축구대표팀이 6월 A 매치 2연전에 나설 명단을 발표했다.

손흥민 (토트넘)과 이강인 (파리생제르맹)등 주축들이 호출됐고 오세훈 (마치다젤비아)과 배준호 (스토크)등 처음 발탁된 선수도 7명이나 이름을 올렸다.

대한축구협회 (KFA)는 27일 2026 FIFA 북중미 월드컵 아시아지역 2차 예선 싱가포르전 (6일), 중국전 (11일)에 나설 6월 A 대표팀 선수 소집 명단 23명을 발표했다.

김도훈 감독 임시체제로 운영되는 이번 소집에는 손흥민, 이강인, 이재성등 기존 주축 멤버들이 변함없이 이름을 올렸다.

동시에 변화도 적잖다.

오세훈, 배준호, 황재원, 최준등 연령별 대표팀에서 활약해온 젊은 선수들과 황인재, 박승욱, 하창래등 연령별 국가대표 경력이 없는 선수까지 총 7명이 A 대표팀에 최초발탁됐다.

이 밖에 부상으로 지난 3월 소집되지 못했던 황희찬과 엄원상이 다시 명단에 포함됐다.

사우디아라비아에서 활약 중인 정우영은 1년 3개월 만에 다시 태극마크를 달았다.

김도훈 감독은 "시간이 촉박한 상황에서 큰 변화를 가져오는 건 어려운 사정이 있었다.

하지만 기존 주축선수들의 부상이나 몸상태 등을 고려할 때 대체자원이 마련돼야 한다는 판단에서 코칭스태프들과 깊이 있는 회의를 통해 명단을 구성했다" 고설명했다.

이어 "새로운 선수들이 상당수 합류했는데 기존의 국가대표 선수들은 기량 면에서 어떤 선수들과도 호흡을 잘 맞출 수 있는 능력이있다.

전술적인 준비를 잘해서 이번 2경기에서 신구 조화가 잘 될 수 있도록 준비를 잘하겠다" 고말했다.

한편 이번 대표팀에는 김민재, 조규성, 설영우등 기존에 꾸준히 발탁됐던 일부 선수들이 부상으로 제외됐다.

김 감독은 "김민재는 최근 리그에서 다친 왼쪽 발목이 아직도 불편함이 있어 정상적인 훈련과 경기출전이 어렵다는 선수 측의 요청이 있었다.

조규성 또한 그동안 통증이 지속됐던 오른쪽 무릎 수술을 이번 소집기간에 진행할 예정이라 선

In [98]:
from langchain_chroma import Chroma

soccer_store = Chroma.from_documents(
    documents=manual_chunks,
    embedding=embeddings,
    ids = [f'article-{i+1}' for i in range(len(manual_chunks))],
    persist_directory='soccer_chroma_db',
    collection_name='soccer_colletion'
)

In [100]:
soccer_retriever = soccer_store.as_retriever(
    search_type='mmr',
    search_kwargs={"k":2,
                   "score_threshold":0.4,
                   "fetch_k":5,
                   },
)

basic_prompt = ChatPromptTemplate.from_template(
    "다음 컨텍스트를 참고해서 질문에 답해줘.\n\n컨텍스트: {context}\n\n질문: {question}"
)


soccer_chain = {"context": soccer_retriever, "question":RunnablePassthrough()} | basic_prompt| o_llm | StrOutputParser()

questions = [
    "선발에서 제외된 선수 이름은?",
"소집 선수는 몇 명이니?",
"수비수로 뽑힌 선수는 모두 몇 명이고, 이름이 뭐니?"
]

for q in questions:
    print("Q:", q)
    print("A:", soccer_chain.invoke(q), "\n")

Q: 선발에서 제외된 선수 이름은?
A: 설영우 선수입니다. 어깨 수술 후 재활 중이라 제외됐습니다. 

Q: 소집 선수는 몇 명이니?
A: 소집 선수는 총 23명입니다. 

Q: 수비수로 뽑힌 선수는 모두 몇 명이고, 이름이 뭐니?
A: 수비수는 **7명**입니다: 권경원, 박승욱, 조유민, 김진수, 이명재, 황재원, 최준. 



In [103]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://ko.wikipedia.org/wiki/인공지능"
wiki_documents = WebBaseLoader(url).load()

text = wiki_documents[0].page_content
text

'\n\n\n\n인공지능 - 위키백과, 우리 모두의 백과사전\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n본문으로 이동\n\n\n\n\n\n\n\n주 메뉴\n\n\n\n\n\n주 메뉴\n사이드바로 이동\n숨기기\n\n\n\n\t\t둘러보기\n\t\n\n\n대문최근 바뀜요즘 화제임의 문서로\n\n\n\n\n\n\t\t사용자 모임\n\t\n\n\n사랑방사용자 모임관리 요청\n\n\n\n\n\n\t\t편집 안내\n\t\n\n\n소개도움말정책과 지침질문방\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n\n검색\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n외관\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n기부\n\n계정 만들기\n\n로그인\n\n\n\n\n\n\n\n\n개인 도구\n\n\n\n\n\n\n기부\n\n\n계정 만들기\n\n\n로그인\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n목차\n사이드바로 이동\n숨기기\n\n\n\n\n처음 위치\n\n\n\n\n\n1\n개념과 범위\n\n\n\n\n개념과 범위 하위 문단 토글하기\n\n\n\n\n\n1.1\n인공지능과 인공지능 시스템\n\n\n\n\n\n\n\n\n1.2\n관련 개념의 관계\n\n\n\n\n\n\n\n\n1.3\n특화형 인공지능과 인공 일반 지능\n\n\n\n\n\n\n\n\n\n\n2\n작동 원리\n\n\n\n\n작동 원리 하위 문단 토글하기\n\n\n\n\n\n2.1\n규칙과 지식에 기반한 방식\n\n\n\n\n\n\n\n\n2.2\n데이터로부터 학습하는 방식\n\n\n\n\n\n\n\n\n2.3\n학습과 추론\n\n\n\n\n\n\n\n\n\n\n3\n주요 능력과 연구 분야\n\n\n\n\n주요 능력과 연구 분야 하위 문단 토글하기\n\n\n\n\n\n3.1\n추론, 문

In [ ]:
# print(modular_rag("인공지능의 역사는 어떻게 되나요?"))
# print(modular_rag("오늘 기분이 어때?"))

from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
    datasource: str = Field(description="'wiki' 또는 'general' 중 하나")

structured_llm = o_llm.with_structured_output(RouteQuery)

route_prompt = ChatPromptTemplate.from_template(
    "다음 질문이 위키피디아 지식 검색이 필요한 질문(wiki)인지, "
    "일반 대화(general)인지 분류해줘.\n질문: {question}"
)
router_chain = route_prompt | structured_llm

def route_and_answer(question: str):
    route = router_chain.invoke({"question": question})
    print('[라우팅 결과] :', route.datasource)
    if route.datasource == "wiki":
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=30)
        docs = text_splitter.split_documents(wiki_documents)

        vector_store = Chroma.from_documents(
            documents=docs,
            ids = [str(i) for i in range(len(docs))],
            embedding=embeddings,
            collection_name='wiki_ai',
            persist_directory='chroma_ai'
        )
        m_retriever = vector_store.as_retriever(
                            search_type = 'mmr',
                            search_kwargs = {"k": 5, 'fetch_k':10}
                        )

        # format_doc
        def format_doc(docs):
            return '\n'.join(doc.page_content for doc in docs)

        # rag_chain

        rag_chain = (
            {'context': m_retriever| format_doc, 'question':RunnablePassthrough()} 
            | prompt 
            | o_llm 
            | StrOutputParser()
        )
        return rag_chain.invoke(question)
    else:
        return o_llm.invoke(question).content

print(route_and_answer("인공지능의 역사는?"))
print(route_and_answer("오늘 기분이 어때?"))

NameError: name 'llm' is not defined